# Tarea C5 — Entrenamiento con Datos Reales
**ECG Rhythm Analyzer | Gabriela — Modelo CNN**  
**Rama:** `feature/model-training`

## Objetivo
Entrenar EfficientNet-B0 con las imágenes reales de ECG generadas por Gloria.
- ~15,000 imágenes de entrenamiento
- Transfer learning en dos fases (igual que C4)
- Guardar checkpoints en Google Drive para no perder progreso

⚠️ **Este entrenamiento puede tardar 2-4 horas en GPU T4. No cierres Colab.**


## PASO 0 — Montar Drive y verificar GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️  Sin GPU. Ve a Entorno de ejecución → Cambiar tipo → GPU T4')

## PASO 1 — Verificar imágenes de Gloria

In [ ]:
import os

# ─── RUTA A LAS IMÁGENES REALES ──────────────────────────────
BASE_DIR = '/content/ecg_images2/ecg_images'

# Si el zip ya fue descomprimido en la sesión anterior, verificar
# Si no existe, descomprimir de nuevo
if not os.path.exists(BASE_DIR):
    print('Descomprimiendo imágenes...')
    os.system('unzip /content/drive/MyDrive/ecg_images_dataset.zip -d /content/ecg_images2')
    print('Listo.')

# Contar imágenes
print('=== Imágenes disponibles ===')
totales = {}
for split in ['train', 'val', 'test']:
    totales[split] = 0
    for clase in ['normal', 'arritmia']:
        path = os.path.join(BASE_DIR, split, clase)
        n = len(os.listdir(path))
        totales[split] += n
        print(f'  {split}/{clase}: {n} imágenes')
    print(f'  → Total {split}: {totales[split]}')
    print()

print(f'TOTAL GENERAL: {sum(totales.values())} imágenes')

## PASO 2 — Crear carpeta de checkpoints en Drive
Los checkpoints se guardan en Drive para no perderlos si Colab se desconecta.

In [ ]:
# Crear carpeta de checkpoints en Google Drive
CHECKPOINT_DIR = '/content/drive/MyDrive/ecg_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'Carpeta de checkpoints: {CHECKPOINT_DIR}')
print('Los modelos se guardarán aquí automáticamente.')

## PASO 3 — Data Loaders con imágenes reales

In [ ]:
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

IMG_SIZE      = 224
BATCH_SIZE    = 32
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Transform de entrenamiento (con augmentation)
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(degrees=5),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 0.5)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# Transform de validación y test (sin augmentation)
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# Datasets
train_dataset = datasets.ImageFolder(os.path.join(BASE_DIR, 'train'), transform=train_transform)
val_dataset   = datasets.ImageFolder(os.path.join(BASE_DIR, 'val'),   transform=val_transform)
test_dataset  = datasets.ImageFolder(os.path.join(BASE_DIR, 'test'),  transform=val_transform)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Clases: {train_dataset.classes}  →  0={train_dataset.classes[0]}, 1={train_dataset.classes[1]}')
print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')
print(f'Batches por epoch (train): {len(train_loader)}')
print(f'Estimado por epoch en GPU T4: ~5-10 minutos')

## PASO 4 — Weighted Loss

In [ ]:
# Calcular pesos según distribución real de clases
n_normal   = len(os.listdir(os.path.join(BASE_DIR, 'train', 'normal')))
n_arritmia = len(os.listdir(os.path.join(BASE_DIR, 'train', 'arritmia')))
total      = n_normal + n_arritmia

weight_normal   = total / (2 * n_normal)
weight_arritmia = total / (2 * n_arritmia)
pos_weight      = torch.tensor([weight_arritmia / weight_normal]).to(DEVICE)
criterion       = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

print('=== Distribución real de clases ===')
print(f'Normal:   {n_normal} ({n_normal/total*100:.1f}%)')
print(f'Arritmia: {n_arritmia} ({n_arritmia/total*100:.1f}%)')
print(f'pos_weight: {pos_weight.item():.4f}')
print('✅ Weighted loss configurado con datos reales.')

## PASO 5 — Modelo y Early Stopping

In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.001, path='mejor_modelo.pt'):
        self.patience  = patience
        self.min_delta = min_delta
        self.path      = path
        self.counter   = 0
        self.best_loss = None
        self.stop      = False

    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            torch.save(model.state_dict(), self.path)
            print(f'    💾 Modelo inicial guardado. Val loss: {val_loss:.4f}')
        elif val_loss < self.best_loss - self.min_delta:
            print(f'    ✅ Mejoró: {self.best_loss:.4f} → {val_loss:.4f}. Guardando modelo.')
            self.best_loss = val_loss
            self.counter   = 0
            torch.save(model.state_dict(), self.path)
        else:
            self.counter += 1
            print(f'    ⚠️  Sin mejora ({self.counter}/{self.patience}). Mejor: {self.best_loss:.4f}')
            if self.counter >= self.patience:
                print(f'    🛑 Early stopping activado.')
                self.stop = True
        return self.stop

print('EarlyStopping definido.')

In [ ]:
# Funciones de entrenamiento
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for batch_idx, (images, labels) in enumerate(loader):
        images = images.to(device)
        labels = labels.float().to(device)
        optimizer.zero_grad()
        outputs = model(images).squeeze()
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        preds = (torch.sigmoid(outputs) > 0.5).float()
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
        if (batch_idx + 1) % 100 == 0:
            print(f'    Batch {batch_idx+1}/{len(loader)} | Loss: {loss.item():.4f}')
    return total_loss / len(loader), correct / total

def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.float().to(device)
            outputs = model(images).squeeze()
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            preds = (torch.sigmoid(outputs) > 0.5).float()
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
    return total_loss / len(loader), correct / total

print('Funciones de entrenamiento definidas.')

In [ ]:
# Crear modelo
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
model.classifier[1] = nn.Linear(1280, 1)
model = model.to(DEVICE)
print('Modelo EfficientNet-B0 listo.')

## PASO 6 — FASE 1: Capas congeladas (10 epochs)

In [ ]:
# Congelar todo excepto classifier
for param in model.parameters():
    param.requires_grad = False
for param in model.classifier.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parámetros entrenables en Fase 1: {trainable:,}')

optimizer_f1  = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
scheduler_f1  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_f1, mode='min', factor=0.5, patience=3)
checkpoint_f1 = os.path.join(CHECKPOINT_DIR, 'modelo_fase1.pt')
early_stop_f1 = EarlyStopping(patience=5, path=checkpoint_f1)

N_EPOCHS_F1 = 10
history_f1  = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

print()
print('=' * 55)
print('FASE 1 — Solo classifier entrenable')
print(f'Epochs: {N_EPOCHS_F1} | LR: 1e-3')
print('=' * 55)

for epoch in range(1, N_EPOCHS_F1 + 1):
    lr = optimizer_f1.param_groups[0]['lr']
    print(f'\n[Epoch {epoch}/{N_EPOCHS_F1}] LR: {lr:.6f}')

    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer_f1, DEVICE)
    val_loss,   val_acc   = val_epoch(model,   val_loader,   criterion, DEVICE)

    history_f1['train_loss'].append(train_loss)
    history_f1['val_loss'].append(val_loss)
    history_f1['train_acc'].append(train_acc)
    history_f1['val_acc'].append(val_acc)

    print(f'  Train → Loss: {train_loss:.4f} | Acc: {train_acc*100:.1f}%')
    print(f'  Val   → Loss: {val_loss:.4f} | Acc: {val_acc*100:.1f}%')

    scheduler_f1.step(val_loss)
    if early_stop_f1(val_loss, model):
        break

print()
print('✅ FASE 1 completada.')
print(f'   Mejor modelo guardado en Drive: {checkpoint_f1}')

## PASO 7 — FASE 2: Fine-tuning (20 epochs)

In [ ]:
# Cargar mejor modelo de Fase 1
model.load_state_dict(torch.load(checkpoint_f1, map_location=DEVICE))
print('Mejor modelo de Fase 1 cargado.')

# Descongelar últimas 3 capas de features + classifier
for param in model.parameters():
    param.requires_grad = False
for bloque in list(model.features.children())[-3:]:
    for param in bloque.parameters():
        param.requires_grad = True
for param in model.classifier.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parámetros entrenables en Fase 2: {trainable:,}')

optimizer_f2  = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5)
scheduler_f2  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_f2, mode='min', factor=0.5, patience=3)
checkpoint_f2 = os.path.join(CHECKPOINT_DIR, 'modelo_fase2.pt')
early_stop_f2 = EarlyStopping(patience=5, path=checkpoint_f2)

N_EPOCHS_F2 = 20
history_f2  = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

print()
print('=' * 55)
print('FASE 2 — Fine-tuning últimas capas')
print(f'Epochs: {N_EPOCHS_F2} | LR: 1e-5')
print('=' * 55)

for epoch in range(1, N_EPOCHS_F2 + 1):
    lr = optimizer_f2.param_groups[0]['lr']
    print(f'\n[Epoch {epoch}/{N_EPOCHS_F2}] LR: {lr:.8f}')

    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer_f2, DEVICE)
    val_loss,   val_acc   = val_epoch(model,   val_loader,   criterion, DEVICE)

    history_f2['train_loss'].append(train_loss)
    history_f2['val_loss'].append(val_loss)
    history_f2['train_acc'].append(train_acc)
    history_f2['val_acc'].append(val_acc)

    print(f'  Train → Loss: {train_loss:.4f} | Acc: {train_acc*100:.1f}%')
    print(f'  Val   → Loss: {val_loss:.4f} | Acc: {val_acc*100:.1f}%')

    scheduler_f2.step(val_loss)
    if early_stop_f2(val_loss, model):
        break

print()
print('✅ FASE 2 completada.')
print(f'   Mejor modelo final guardado en Drive: {checkpoint_f2}')

## PASO 8 — Curvas de entrenamiento

In [ ]:
import matplotlib.pyplot as plt

e1 = list(range(1, len(history_f1['train_loss']) + 1))
e2 = list(range(len(e1) + 1, len(e1) + len(history_f2['train_loss']) + 1))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Curvas de Entrenamiento — Datos Reales ECG', fontsize=12)

axes[0].plot(e1, history_f1['train_loss'], 'b-o', label='Train F1')
axes[0].plot(e1, history_f1['val_loss'],   'b--o', label='Val F1')
axes[0].plot(e2, history_f2['train_loss'], 'r-o', label='Train F2')
axes[0].plot(e2, history_f2['val_loss'],   'r--o', label='Val F2')
axes[0].axvline(x=len(e1)+0.5, color='gray', linestyle=':', label='Inicio F2')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(e1, [a*100 for a in history_f1['train_acc']], 'b-o', label='Train F1')
axes[1].plot(e1, [a*100 for a in history_f1['val_acc']],   'b--o', label='Val F1')
axes[1].plot(e2, [a*100 for a in history_f2['train_acc']], 'r-o', label='Train F2')
axes[1].plot(e2, [a*100 for a in history_f2['val_acc']],   'r--o', label='Val F2')
axes[1].axvline(x=len(e1)+0.5, color='gray', linestyle=':', label='Inicio F2')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINT_DIR, 'curvas_entrenamiento_real.png'), dpi=100, bbox_inches='tight')
plt.show()
print('Gráfica guardada en Drive.')

## ✅ C5 Completada
**Próximo paso:** Notebook C6 — Evaluación en test set + exportar ONNX para Cristina.